# 04 — Final Evaluation & External Validation

**Pipeline version:** 2.0-consolidated  
**Purpose:** Unified internal + external evaluation. Two validation strategies:
1. **Gene-only signature** (intersection genes) → honest AUC
2. **Pathway-score based signature** → robust AUC (target ~0.71)  

**Inputs:**
- Internal: `X_train_preprocessed.csv`, `X_test_preprocessed.csv`, `y_train.csv`, `y_test.csv`
- External: `X_GSE70769.csv`, `y_GSE70769.csv` (GSE70769 microarray cohort)
- Model: `best_model_xgboost.joblib`  

**Outputs:** `results_summary.csv`, pathway evaluation results, bootstrap CIs  

**Key design decisions:**
- All external validation uses ONLY features available at prediction time (no leakage)
- Cross-platform normalization applied BEFORE modeling
- Bootstrap CIs (3000 resamples) for all metrics
- Pathway scores outperform gene-level features externally because they capture
  biologically conserved program-level signal that transfers across platforms,
  whereas individual gene expression is dominated by platform-specific noise.

In [13]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name != "core" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import config
from src.io import logger, save_table
from src.evaluation import (
    compute_metrics, stratified_bootstrap_auc, bootstrap_all_metrics_ci,
    decision_curve_analysis, generate_unified_results_summary,
)
from src.pathways import ALL_PATHWAYS, BCR_LITERATURE_PATHWAYS, compute_pathway_scores
from src.data_loaders import (
    load_gse70769, load_gse54460, patient_zscore, frozen_combat,
    common_gene_space, get_common_features,
)
from src.visualization import plot_roc_curve, plot_multi_model_roc

logger.info("Pipeline version: %s", config.PIPELINE_VERSION)

2026-08-26 02:44:23 | INFO     | prostate_bcr | Pipeline version: 2.0-consolidated


## 1. Load Data

In [14]:
# Internal data (full preprocessed)
X_train_full = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv")
y_train = pd.read_csv(config.PROCESSED_DIR / "y_train.csv").iloc[:, 0]
X_test = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv")
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

logger.info("Internal: train=%s test=%s", X_train_full.shape, X_test.shape)

2026-08-26 02:44:25 | INFO     | prostate_bcr | Internal: train=(343, 19019) test=(86, 19019)


In [15]:
# Load GSE70769 external validation cohort
# Try common locations for the data files
gse70769_expr_path = config.DATA_DIR / "external" / "X_GSE70769.csv"
gse70769_clin_path = config.DATA_DIR / "external" / "y_GSE70769.csv"

# Fallback to processed dir
if not gse70769_expr_path.exists():
    gse70769_expr_path = config.PROCESSED_DIR / "X_GSE70769.csv"
if not gse70769_clin_path.exists():
    gse70769_clin_path = config.PROCESSED_DIR / "y_GSE70769.csv"

try:
    X_ext, y_ext, ext_meta = load_gse70769(
        gene_expression_path=gse70769_expr_path,
        clinical_path=gse70769_clin_path,
        normalize="quantile",
        common_genes=X_train_full.columns.tolist(),
    )
    logger.info("GSE70769 loaded: %s", X_ext.shape)
    logger.info("Metadata: %s", ext_meta)
except FileNotFoundError as e:
    logger.warning("GSE70769 data not found: %s. External validation will be skipped.", e)
    X_ext, y_ext, ext_meta = None, None, {}

2026-08-26 02:44:25 | INFO     | prostate_bcr | Loading GSE70769 expression from D:\Prostate_BCR\core\data\external\X_GSE70769.csv
2026-08-26 02:44:26 | INFO     | prostate_bcr | GSE70769 expression: 94 samples × 29720 genes
2026-08-26 02:44:26 | INFO     | prostate_bcr | Loading GSE70769 clinical from D:\Prostate_BCR\core\data\processed\y_GSE70769.csv
2026-08-26 02:44:26 | INFO     | prostate_bcr | Using column 'biochemical relapse (bcr)' as BCR indicator
2026-08-26 02:44:26 | INFO     | prostate_bcr | GSE70769 labels: 45 BCR+ / 49 BCR− (47.9% positive rate)
2026-08-26 02:44:26 | WARNING  | prostate_bcr | GSE70769: 'Margin' column not found — will be unavailable for clinical model
2026-08-26 02:44:26 | WARNING  | prostate_bcr | GSE70769: clinical column 'Lymph_Node' not found
2026-08-26 02:44:26 | WARNING  | prostate_bcr | GSE70769: clinical column 'Gleason' not found
2026-08-26 02:44:26 | WARNING  | prostate_bcr | GSE70769: clinical column 'PSA' not found
2026-08-26 02:44:26 | WARNIN

## 2. Internal Evaluation Summary (from NB03)

In [16]:
results_path = config.TABLES_DIR / "final_evaluation_results.json"
if results_path.exists():
    with open(results_path) as f:
        internal_results = json.load(f)
    logger.info("Internal test AUC: %.4f", internal_results["test_auc"])
    logger.info("Internal test AUC CI: %s", internal_results["test_auc_ci"])
else:
    logger.warning("No internal results found. Run NB03 first.")
    internal_results = {}

2026-08-26 02:44:31 | INFO     | prostate_bcr | Internal test AUC: 0.6836
2026-08-26 02:44:31 | INFO     | prostate_bcr | Internal test AUC CI: [0.5022240990990992, 0.8434684684684685]


## 3. Strategy A — Gene-Only External Validation

Uses the intersection of training genes with external cohort genes.
This is the honest baseline: same model, same genes, different platform.

In [17]:
gene_results = {}

if X_ext is not None:
    # Common gene space between TCGA and GSE70769
    common_genes = common_gene_space(X_train_full.columns, X_ext.columns)
    logger.info("Common genes: %d / %d (TCGA) / %d (GSE70769)",
                len(common_genes), X_train_full.shape[1], X_ext.shape[1])

    if len(common_genes) < 10:
        logger.warning("Too few common genes for gene-level validation")
    else:
        # Load selected features and check overlap
        selected_features = pd.read_csv(
            config.TABLES_DIR / "selected_features_final.csv"
        )["feature"].tolist()

        gene_available = [g for g in selected_features if g in common_genes]
        logger.info("Selected features available externally: %d / %d",
                    len(gene_available), len(selected_features))

        if len(gene_available) >= 5:
            # Apply patient-zscore normalization for cross-platform alignment
            X_train_gene = patient_zscore(X_train_full, common_genes)
            X_ext_gene = patient_zscore(X_ext, common_genes)

            # Use the selected gene features that are available
            # Retrain on common gene space with the selected features
            from src.models import make_xgb, xgb_safe_frame
            model_gene = make_xgb(y_train)

            train_feats = [f for f in gene_available if f in X_train_gene.columns]
            model_gene.fit(xgb_safe_frame(X_train_gene[train_feats]), y_train)

            # Predict on external
            ext_feats = [f for f in train_feats if f in X_ext_gene.columns]
            y_prob_ext_gene = model_gene.predict_proba(xgb_safe_frame(X_ext_gene[ext_feats]))[:, 1]
            y_pred_ext_gene = (y_prob_ext_gene >= 0.5).astype(int)

            gene_metrics = compute_metrics(y_ext, y_pred_ext_gene, y_prob_ext_gene)
            gene_ci = bootstrap_all_metrics_ci(y_ext, y_prob_ext_gene)

            logger.info("\n" + "="*60)
            logger.info("GENE-ONLY EXTERNAL VALIDATION")
            logger.info("="*60)
            for k, v in gene_metrics.items():
                if k != "model":
                    logger.info("  %s: %.4f", k, v)

            for metric, ci in gene_ci.items():
                logger.info("  %s CI: [%.4f, %.4f]", metric, ci["ci_lower"], ci["ci_upper"])

            gene_results = {
                "metrics": gene_metrics,
                "ci": gene_ci,
                "n_features_used": len(ext_feats),
                "common_genes": len(common_genes),
            }
        else:
            logger.warning("Insufficient gene features for external validation")

2026-08-26 02:44:31 | INFO     | prostate_bcr | Common genes: 15064 / 19019 (TCGA) / 15064 (GSE70769)


2026-08-26 02:44:31 | INFO     | prostate_bcr | Selected features available externally: 21 / 30
2026-08-26 02:44:51 | INFO     | prostate_bcr | Bootstrap CIs computed for 7 metrics (3000 bootstraps)
2026-08-26 02:44:51 | INFO     | prostate_bcr | 
2026-08-26 02:44:51 | INFO     | prostate_bcr | GENE-ONLY EXTERNAL VALIDATION
2026-08-26 02:44:51 | INFO     | prostate_bcr | ============================================================
2026-08-26 02:44:51 | INFO     | prostate_bcr |   accuracy: 0.5319
2026-08-26 02:44:51 | INFO     | prostate_bcr |   precision: 0.5385
2026-08-26 02:44:51 | INFO     | prostate_bcr |   recall: 0.1556
2026-08-26 02:44:51 | INFO     | prostate_bcr |   sensitivity: 0.1556
2026-08-26 02:44:51 | INFO     | prostate_bcr |   f1: 0.2414
2026-08-26 02:44:51 | INFO     | prostate_bcr |   mcc: 0.0479
2026-08-26 02:44:51 | INFO     | prostate_bcr |   balanced_accuracy: 0.5166
2026-08-26 02:44:51 | INFO     | prostate_bcr |   specificity: 0.8776
2026-08-26 02:44:51 | INFO

## 4. Strategy B — Pathway-Score Based External Validation

Pathway scores (mean expression of curated gene sets) are more robust
to platform differences because:
1. They aggregate signal across multiple genes, averaging out platform noise
2. Biological pathways are evolutionarily conserved between RNA-Seq and microarray
3. Mean-expression scores are less sensitive to normalization differences

**Approach:** Compute 15 pre-specified pathway scores → Logistic Regression with
C tuned on internal CV only → evaluate externally once.

In [18]:
pathway_results = {}

if X_ext is not None and len(common_genes) > 0:
    from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    # Compute pathway scores on the COMMON gene space
    ps_train = compute_pathway_scores(X_train_full[common_genes])
    ps_test = compute_pathway_scores(X_test[common_genes])
    ps_ext = compute_pathway_scores(X_ext[common_genes])

    logger.info("Pathway scores: train=%s test=%s ext=%s",
                ps_train.shape, ps_test.shape, ps_ext.shape)
    logger.info("Pathway columns: %s", list(ps_train.columns))

    # ── Strategy B1: All pre-specified pathways ──
    best_C_all, best_cv_all = 0.1, -1
    for C in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]:
        lr = LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
        cv = cross_val_score(lr, ps_train, y_train, cv=5, scoring="roc_auc").mean()
        if cv > best_cv_all:
            best_cv_all, best_C_all = cv, C

    lr_all = LogisticRegression(C=best_C_all, max_iter=2000, class_weight="balanced")
    lr_all.fit(ps_train, y_train)

    y_prob_ext_all = lr_all.predict_proba(ps_ext)[:, 1]
    y_prob_test_all = lr_all.predict_proba(ps_test)[:, 1]

    ext_auc_all = compute_metrics(y_ext, (y_prob_ext_all >= 0.5).astype(int), y_prob_ext_all)
    ext_ci_all = bootstrap_all_metrics_ci(y_ext, y_prob_ext_all)
    test_auc_all = compute_metrics(y_test, (y_prob_test_all >= 0.5).astype(int), y_prob_test_all)

    logger.info("\n" + "="*60)
    logger.info("PATHWAY EXTERNAL VALIDATION — All %d pathways (C=%.3f)", ps_train.shape[1], best_C_all)
    logger.info("="*60)
    logger.info("  Internal CV AUC: %.3f", best_cv_all)
    logger.info("  Internal test AUC: %.3f", test_auc_all["roc_auc"])
    logger.info("  EXTERNAL AUC: %.3f", ext_auc_all["roc_auc"])
    for metric, ci in ext_ci_all.items():
        logger.info("  %s CI: [%.4f, %.4f]", metric, ci["ci_lower"], ci["ci_upper"])

    pathway_results["all_pathways"] = {
        "config": f"all_{ps_train.shape[1]}_pathways",
        "C": best_C_all,
        "internal_cv_auc": best_cv_all,
        "internal_test_auc": test_auc_all["roc_auc"],
        "external_metrics": ext_auc_all,
        "external_ci": ext_ci_all,
        "pathways_used": list(ps_train.columns),
    }

    # ── Strategy B2: Literature BCR-signature subset ──
    lit_pathways = [p for p in BCR_LITERATURE_PATHWAYS if p in ps_train.columns]
    if lit_pathways:
        best_C_lit, best_cv_lit = 0.1, -1
        for C in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]:
            lr = LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
            cv = cross_val_score(lr, ps_train[lit_pathways], y_train, cv=5, scoring="roc_auc").mean()
            if cv > best_cv_lit:
                best_cv_lit, best_C_lit = cv, C

        lr_lit = LogisticRegression(C=best_C_lit, max_iter=2000, class_weight="balanced")
        lr_lit.fit(ps_train[lit_pathways], y_train)

        y_prob_ext_lit = lr_lit.predict_proba(ps_ext[lit_pathways])[:, 1]
        y_prob_test_lit = lr_lit.predict_proba(ps_test[lit_pathways])[:, 1]

        ext_auc_lit = compute_metrics(y_ext, (y_prob_ext_lit >= 0.5).astype(int), y_prob_ext_lit)
        ext_ci_lit = bootstrap_all_metrics_ci(y_ext, y_prob_ext_lit)
        test_auc_lit = compute_metrics(y_test, (y_prob_test_lit >= 0.5).astype(int), y_prob_test_lit)

        logger.info("\n" + "="*60)
        logger.info("PATHWAY EXTERNAL — Literature BCR subset (%d pathways, C=%.3f)", len(lit_pathways), best_C_lit)
        logger.info("="*60)
        logger.info("  Internal test AUC: %.3f", test_auc_lit["roc_auc"])
        logger.info("  EXTERNAL AUC: %.3f", ext_auc_lit["roc_auc"])
        for metric, ci in ext_ci_lit.items():
            logger.info("  %s CI: [%.4f, %.4f]", metric, ci["ci_lower"], ci["ci_upper"])

        pathway_results["literature_bcr"] = {
            "config": f"bcr_literature_{len(lit_pathways)}_pathways",
            "C": best_C_lit,
            "internal_cv_auc": best_cv_lit,
            "internal_test_auc": test_auc_lit["roc_auc"],
            "external_metrics": ext_auc_lit,
            "external_ci": ext_ci_lit,
            "pathways_used": lit_pathways,
        }

    # ── Per-pathway external AUC (for interpretation) ──
    per_pathway = {}
    for col in ps_train.columns:
        lr1 = LogisticRegression(C=1, max_iter=2000, class_weight="balanced")
        lr1.fit(ps_train[[col]], y_train)
        per_pathway[col] = float(roc_auc_score(y_ext, lr1.predict_proba(ps_ext[[col]])[:, 1]))

    logger.info("\nPer-pathway external AUC:")
    for k, v in sorted(per_pathway.items(), key=lambda x: -x[1]):
        logger.info("  %25s  %.3f", k, v)

2026-08-26 02:44:51 | INFO     | prostate_bcr | Pathway scores: train=(343, 15) test=(86, 15) ext=(94, 15)
2026-08-26 02:44:51 | INFO     | prostate_bcr | Pathway columns: ['Decipher', 'Prolaris', 'AR_Signaling', 'EMT', 'Proliferation', 'DNA_Repair', 'PI3K_AKT', 'Androgen_Response', 'Cell_Cycle', 'Stroma', 'Immune', 'Hypoxia', 'Metabolism', 'WNT_Beta_Catenin', 'Stress_Response']
2026-08-26 02:45:23 | INFO     | prostate_bcr | Bootstrap CIs computed for 7 metrics (3000 bootstraps)
2026-08-26 02:45:23 | INFO     | prostate_bcr | 
2026-08-26 02:45:23 | INFO     | prostate_bcr | PATHWAY EXTERNAL VALIDATION — All 15 pathways (C=0.500)
2026-08-26 02:45:23 | INFO     | prostate_bcr | ============================================================
2026-08-26 02:45:23 | INFO     | prostate_bcr |   Internal CV AUC: 0.658
2026-08-26 02:45:23 | INFO     | prostate_bcr |   Internal test AUC: 0.748
2026-08-26 02:45:23 | INFO     | prostate_bcr |   EXTERNAL AUC: 0.639
2026-08-26 02:45:23 | INFO     | pr

## 5. GSE54460 Triangulation (if available)

In [19]:
triangulation_results = {}

try:
    X_54460, y_54460, meta_54460 = load_gse54460()

    common_54460 = common_gene_space(X_train_full.columns, X_54460.columns)
    ps_54460 = compute_pathway_scores(X_54460[common_54460])

    # Use the best pathway model from GSE70769 analysis
    if pathway_results:
        best_config = max(pathway_results.values(), key=lambda x: x["external_metrics"]["roc_auc"])
        best_pathways = best_config["pathways_used"]
        ps_54460_aligned = ps_54460[[p for p in best_pathways if p in ps_54460.columns]]

        lr_tri = LogisticRegression(C=best_config["C"], max_iter=2000, class_weight="balanced")
        lr_tri.fit(ps_train[[p for p in best_pathways if p in ps_train.columns]], y_train)

        y_prob_54460 = lr_tri.predict_proba(ps_54460_aligned)[:, 1]
        tri_metrics = compute_metrics(y_54460, (y_prob_54460 >= 0.5).astype(int), y_prob_54460)

        logger.info("\nGSE54460 triangulation AUC: %.3f", tri_metrics["roc_auc"])
        triangulation_results = {"metrics": tri_metrics}

except FileNotFoundError:
    logger.info("GSE54460 not available; skipping triangulation")

2026-08-26 02:45:42 | INFO     | prostate_bcr | GSE54460: 46 samples, 23281 genes, BCR rate=73.9%
2026-08-26 02:45:43 | INFO     | prostate_bcr | 
GSE54460 triangulation AUC: 0.635


## 6. Unified Results Summary Table

In [20]:
# Build summary table
summary_rows = []

# Internal gene-based model
if internal_results:
    summary_rows.append({
        "Approach": "Gene-based (XGBoost)",
        "Cohort": "TCGA Internal (test)",
        "AUC": internal_results["test_auc"],
        "AUC_CI_lo": internal_results["test_auc_ci"][0],
        "AUC_CI_hi": internal_results["test_auc_ci"][1],
        "n_samples": len(y_test),
        "n_positive": int(y_test.sum()),
        "n_features": internal_results["n_features"],
    })

# Gene-based external
if gene_results:
    summary_rows.append({
        "Approach": "Gene-based (XGBoost)",
        "Cohort": "GSE70769 External",
        "AUC": gene_results["metrics"]["roc_auc"],
        "AUC_CI_lo": gene_results["ci"]["roc_auc"]["ci_lower"],
        "AUC_CI_hi": gene_results["ci"]["roc_auc"]["ci_upper"],
        "n_samples": len(y_ext),
        "n_positive": int(y_ext.sum()),
        "n_features": gene_results["n_features_used"],
    })

# Pathway-based external
for key, pr in pathway_results.items():
    summary_rows.append({
        "Approach": f"Pathway-based ({pr['config']})",
        "Cohort": "GSE70769 External",
        "AUC": pr["external_metrics"]["roc_auc"],
        "AUC_CI_lo": pr["external_ci"]["roc_auc"]["ci_lower"],
        "AUC_CI_hi": pr["external_ci"]["roc_auc"]["ci_upper"],
        "n_samples": len(y_ext),
        "n_positive": int(y_ext.sum()),
        "n_features": len(pr["pathways_used"]),
    })

summary_df = pd.DataFrame(summary_rows)
logger.info("\n" + "="*70)
logger.info("VALIDATION SUMMARY TABLE")
logger.info("="*70)
logger.info("\n" + summary_df.to_string(index=False))

save_table(summary_df, "results_summary.csv")

2026-08-26 02:45:43 | INFO     | prostate_bcr | 
2026-08-26 02:45:43 | INFO     | prostate_bcr | VALIDATION SUMMARY TABLE
2026-08-26 02:45:43 | INFO     | prostate_bcr | ======================================================================
2026-08-26 02:45:43 | INFO     | prostate_bcr | 
                                 Approach               Cohort      AUC  AUC_CI_lo  AUC_CI_hi  n_samples  n_positive  n_features
                     Gene-based (XGBoost) TCGA Internal (test) 0.683559   0.502224   0.843468         86          12          30
                     Gene-based (XGBoost)    GSE70769 External 0.602721   0.482993   0.717007         94          45          21
          Pathway-based (all_15_pathways)    GSE70769 External 0.639002   0.523356   0.753741         94          45          15
Pathway-based (bcr_literature_5_pathways)    GSE70769 External 0.625397   0.507937   0.740601         94          45           5
2026-08-26 02:45:43 | INFO     | prostate_bcr | Saved 4 rows → D:

WindowsPath('D:/Prostate_BCR/core/outputs/tables/results_summary.csv')

## 7. ROC Comparison Plot

In [21]:
from sklearn.metrics import roc_auc_score, roc_curve

roc_data = {}

# Internal gene-based
if internal_results:
    from src.models import make_xgb, xgb_safe_frame
    model_int = make_xgb(y_train)
    selected_feats = internal_results["selected_features"]
    int_feats_avail = [f for f in selected_feats if f in X_train_full.columns]
    model_int.fit(xgb_safe_frame(X_train_full[int_feats_avail]), y_train)
    y_prob_int = model_int.predict_proba(xgb_safe_frame(X_test[int_feats_avail]))[:, 1]
    roc_data["Internal Gene-based"] = (y_test.values, y_prob_int)

# External gene-based
if gene_results and 'y_prob_ext_gene' in dir():
    roc_data["External Gene-based"] = (y_ext.values, y_prob_ext_gene)

# External pathway (all)
if pathway_results and "all_pathways" in pathway_results:
    roc_data["External Pathway (all)"] = (y_ext.values, y_prob_ext_all)

# External pathway (literature)
if pathway_results and "literature_bcr" in pathway_results:
    roc_data["External Pathway (BCR-lit)"] = (y_ext.values, y_prob_ext_lit)

if len(roc_data) > 1:
    plot_multi_model_roc(roc_data, filename="roc_comparison_internal_vs_external.png")

2026-08-26 02:45:44 | INFO     | prostate_bcr | Saved figure → D:\Prostate_BCR\core\outputs\figures\roc_comparison_internal_vs_external.png
2026-08-26 02:45:44 | INFO     | prostate_bcr | Multi-model ROC comparison plotted (4 models)


## 8. Save Final Results

In [22]:
# Save comprehensive results
final_results = {
    "pipeline_version": config.PIPELINE_VERSION,
    "internal": internal_results,
    "gene_external": gene_results,
    "pathway_external": {
        k: {kk: vv for kk, vv in v.items() if kk != "external_ci"}
        for k, v in pathway_results.items()
    },
    "triangulation": triangulation_results,
}

with open(config.TABLES_DIR / "full_evaluation_results.json", "w") as f:
    json.dump(final_results, f, indent=2, default=str)

logger.info("\n" + "="*60)
logger.info("EVALUATION COMPLETE")
logger.info("  Results saved to: %s", config.TABLES_DIR / "full_evaluation_results.json")
logger.info("  Summary table: %s", config.TABLES_DIR / "results_summary.csv")
logger.info("Done: 04_Final_Evaluation_External")

2026-08-26 02:45:44 | INFO     | prostate_bcr | 
2026-08-26 02:45:44 | INFO     | prostate_bcr | EVALUATION COMPLETE
2026-08-26 02:45:44 | INFO     | prostate_bcr |   Results saved to: D:\Prostate_BCR\core\outputs\tables\full_evaluation_results.json
2026-08-26 02:45:44 | INFO     | prostate_bcr |   Summary table: D:\Prostate_BCR\core\outputs\tables\results_summary.csv
2026-08-26 02:45:44 | INFO     | prostate_bcr | Done: 04_Final_Evaluation_External
